# 🚀 Notebook 05 — Model Deployment & Batch Scoring

**Goal:** Register the best model in MLflow Model Registry and batch score all 500 accounts. Results saved to `gold_credit_risk_scores` Delta Table.

> **Run time:** ~5 min

In [ ]:
import mlflow
import mlflow.sklearn
from mlflow import MlflowClient
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder

client = MlflowClient()
model_name = 'CreditRiskScorer'

# Find the best run by AUC-ROC
runs = mlflow.search_runs(
    experiment_names=['CreditRiskScoring'],
    order_by=['metrics.auc_roc DESC']
)
best_run = runs.iloc[0]
print(f'Best model: {best_run["tags.mlflow.runName"]}')
print(f'AUC-ROC:    {best_run["metrics.auc_roc"]:.4f}')
print(f'Run ID:     {best_run["run_id"]}')

## Step 1 — Register Model in MLflow Model Registry

In [ ]:
# Register the best model
model_uri = f"runs:/{best_run['run_id']}/{best_run['tags.mlflow.runName'].lower().replace(' ','_')}"

registered = mlflow.register_model(model_uri=model_uri, name=model_name)
print(f'\n✅ Registered model: {registered.name} v{registered.version}')

# Transition to Production
client.transition_model_version_stage(
    name=model_name,
    version=registered.version,
    stage='Production',
    archive_existing_versions=True
)
print(f'✅ Model v{registered.version} promoted to Production')

## Step 2 — Load Production Model

In [ ]:
# Load directly from Model Registry
model = mlflow.pyfunc.load_model(f'models:/{model_name}/Production')
print(f'Loaded: {model_name} (Production)')

## Step 3 — Batch Score All 500 Accounts

In [ ]:
# Prepare full dataset for scoring
df_all = spark.table('silver_credit_risk_features').toPandas()

cat_cols = ['AccountType','Branch','Status','LoanPurpose','EmploymentType','HomeOwnership']
for col in cat_cols:
    le = LabelEncoder()
    df_all[col + '_enc'] = le.fit_transform(df_all[col].astype(str))

feature_cols = [
    'CreditScore','AnnualIncome','LoanAmount','EmploymentYears',
    'DebtToIncomeRatio','NumOpenAccounts','NumDelinquencies',
    'MonthsSinceLastDelinquency','LoanTermMonths',
    'TotalTransactions','TotalLoanAmount','AvgTransactionAmount',
    'NumLoans','MaxTransactionAmount',
    'AccountType_enc','LoanPurpose_enc','EmploymentType_enc','HomeOwnership_enc'
]
X_all = df_all[feature_cols].fillna(0)

# Score
scores = model.predict(X_all)
print(f'Scored {len(scores)} accounts')
print(f'Score distribution: min={scores.min():.3f} mean={scores.mean():.3f} max={scores.max():.3f}')

## Step 4 — Save Scores to Gold Table

In [ ]:
import pandas as pd
from datetime import datetime

# Build output dataframe
df_scores = pd.DataFrame({
    'AccountID':        df_all['AccountID'],
    'DefaultProbability': np.round(scores, 4),
    'RiskCategory': pd.cut(
        scores,
        bins=[0, 0.2, 0.4, 0.6, 1.0],
        labels=['Low Risk','Medium Risk','High Risk','Very High Risk']
    ).astype(str),
    'ActualDefault': df_all['IsDefault'],
    'ScoredAt':      datetime.utcnow().strftime('%Y-%m-%dT%H:%M:%SZ'),
    'ModelVersion':  f'v{registered.version}',
    'ModelName':     model_name
})

# Save to Gold Delta Table
spark.createDataFrame(df_scores) \
     .write.format('delta').mode('overwrite') \
     .saveAsTable('gold_credit_risk_scores')

print('✅ gold_credit_risk_scores saved')
print('\nRisk Category Distribution:')
print(df_scores['RiskCategory'].value_counts().sort_index())

## Step 5 — Query Gold Table

In [ ]:
%%sql
SELECT
    RiskCategory,
    COUNT(*) AS Accounts,
    ROUND(AVG(DefaultProbability) * 100, 1) AS AvgDefaultProb_Pct,
    SUM(ActualDefault) AS ActualDefaults
FROM gold_credit_risk_scores
GROUP BY RiskCategory
ORDER BY AvgDefaultProb_Pct DESC